# 하수 바이러스 활동 예측 — 시계열 EDA와 모델 (CDC NWSS)

- 데이터: 미국 하수처리장 주간 바이러스 수준 (23,301행 · 40지점)
- 목표: 하수 바이러스 활동 수준(WVAL) 예측 — 다중 시계열
- 흐름: 불러오기 → 학습 전 확인 → 시계열 EDA → 형식 변환 → 학습 → 해석
- 참고: 데이터 소개 CDC_NWSS_wastewater.txt

- 이 데이터의 핵심
  · **병원체 3종이 한 컬럼에 섞여 있음** → 필터 필수
  · 주간(토요일 종료) 데이터 · 겨울 유행 계절성
  · 하수는 임상 검사보다 먼저 유행을 탐지 (조기 경보)

## 1. 불러오기

- 이미 숫자 변환 완료된 축소본 (원본은 쉼표 문자열)
- 지점 40개 × 병원체 3종

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("nwss_wastewater_40sites.csv",
                 parse_dates=["Week_End"])
print(df.shape)                    # (23301, 7)
print(df.columns.tolist())
df.head()

## 2. 학습 전 확인

### 2-1. 병원체가 한 컬럼에 섞여 있음 (핵심)

- 같은 지점·같은 주에 병원체별로 행이 3개
- 필터 없이 쓰면 3종이 한 시계열에 뒤섞임
- 병원체 하나를 선택해야 함

In [ ]:
print(df["Pathogen_Target"].value_counts())

# 독감 선택 (겨울 유행 뚜렷)
sub = df[df["Pathogen_Target"] == "Influenza A virus"].copy()
print("\n독감만:", sub.shape, "| 지점:", sub["Site"].nunique())

### 2-2. 3열 형식으로 정리

- Site = item_id, Week_End = timestamp, Site_WVAL = target

In [ ]:
ts_df = sub.rename(columns={
    "Site": "item_id",
    "Week_End": "timestamp",
    "Site_WVAL": "target",
})[["item_id", "timestamp", "target"]]
print(ts_df.head())

### 2-3. 지점별 관측 기간 다름

- 지점이 시기마다 들어오고 나감 (신규·중단)

In [ ]:
period = ts_df.groupby("item_id")["timestamp"].agg(["min","max","size"])
print(period.head(10))
print("\n관측 주차 범위:", period["size"].min(), "~", period["size"].max())

## 3. 시계열 EDA

### 3-1. 계절성 — 겨울 유행

- 독감은 겨울에 유행 → 연 주기 예상

In [ ]:
(ts_df.assign(m=ts_df["timestamp"].dt.month)
   .groupby("m")["target"].mean()
   .plot(kind="bar", figsize=(8,3), title="mean WVAL by month (Influenza)"))
plt.show()
# 겨울(12~2월) 높고 여름 낮은 패턴

### 3-2. 특정 지점의 추이

- 해마다 겨울 피크가 반복되는가

In [ ]:
one = ts_df["item_id"].iloc[0]
s = ts_df[ts_df["item_id"]==one].set_index("timestamp")["target"]

s.plot(figsize=(12,3), title=f"WVAL trend - {one}")
plt.show()

### 3-3. target 분포 — 치우침

- 평소 낮고 유행기에 급등 → 분포가 치우침

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,3))
ts_df["target"].hist(bins=60, ax=ax[0]); ax[0].set_title("WVAL (raw)")
np.log1p(ts_df["target"]).hist(bins=60, ax=ax[1])
ax[1].set_title("log1p(WVAL)")
plt.show()
# 치우침이 크면 로그 변환 검토

## 4. 시계열 형식 변환

- 주간 데이터 (토요일 종료) → freq="W-SAT" 

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame

ts = TimeSeriesDataFrame.from_data_frame(
    ts_df, id_column="item_id", timestamp_column="timestamp")
ts = ts.convert_frequency(freq="W-SAT")   # 빈 주 채움
print("변환 완료:", ts.shape)

## 5. 학습

- prediction_length = 8 (약 2개월 앞)
- 40개 지점을 함께 학습 (전역 모델)

In [ ]:
from autogluon.timeseries import TimeSeriesPredictor

prediction_length = 8
train_data, test_data = ts.train_test_split(prediction_length)

predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="target",
    freq="W-SAT",
).fit(train_data, presets="medium_quality", time_limit=600)

In [ ]:
predictor.leaderboard(test_data)
# SeasonalNaive 순위 확인 — 겨울 유행이 규칙적이면 상위

## 6. 해석

In [ ]:
predictions = predictor.predict(train_data)

predictor.plot(
    data=test_data,
    predictions=predictions,
    item_ids=[one],
    quantile_levels=[0.1, 0.9],
    max_history_length=52,
)
plt.show()

### 6-1. 생각해 볼 점

- 유행 시작 시점을 미리 맞힐 수 있는가
  → 계절 패턴은 잡아도, 그 해 유행 강도는 예측이 어려움
- 40개 지점을 함께 학습(전역)한 것이 유리했는가
  → 축소본은 위스콘신·캘리포니아에 집중
  → 같은 주 지점끼리 유행 시점이 비슷 → 전역 모델에 유리할 수 있음
- 다른 병원체(SARS-CoV-2, RSV)로 바꾸면 결과가 달라지는가
  → SARS는 대유행기 단절, RSV는 여름 데이터 적음

- 조기 경보 관점: 하수 신호가 며칠~몇 주 앞서는가
  → 임상 데이터와 함께 보면 확인 가능 (이 실습 범위 밖)

## 정리

- 병원체 3종이 한 컬럼 → 반드시 하나만 필터
- 주간 데이터(토요일 종료) → freq="W-SAT"
- 지점별 기간 다름 → convert_frequency로 빈 주 채움
- 겨울 유행 계절성 · 치우친 분포(로그 변환 검토)
- 40개 지점 전역 모델

- 이 데이터의 교훈: 한 컬럼에 여러 시계열이 섞여 있으면
  먼저 나눠야 한다 → 데이터 구조 파악이 첫 단계